In [18]:
import prefect as pf
import pandas as pd
from pymongo import MongoClient
import requests
from dotenv import load_dotenv
import os

# Llamar API

def fetch_countries_data():
    url = "https://restcountries.com/v3.1/region/africa"
    params = {}

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        print(f"Successfully fetched {len(data)} countries")
        return data
    except requests.exceptions.HTTPError as e:
        print(f"HTTP error: {e}")
    except requests.exceptions.ConnectionError:
        print("Connection error: could not reach the API")
    except requests.exceptions.Timeout:
        print("Request timed out")
    except requests.exceptions.RequestException as e:
        print(f"Unexpected error: {e}")

    return None

# Cargar datos API a un dataframe local

def load_to_dataframe(data):
    df = pd.json_normalize(data)
    print(df.head())
    print(df.shape)
    return df

# Transform datos

def transform_data(df):
    # Select relevant columns
    cols_to_keep = [
        'name.common', 'cca3', 'capital', 'region', 'subregion',
        'population', 'area', 'landlocked', 'borders', 'latlng',
        'flag', 'idd.root', 'idd.suffixes'
    ]

    # Add dynamic columns (languages, currencies, gini)
    language_cols = df.filter(like='languages.').columns.tolist()
    currency_cols = df.filter(like='currencies.').columns.tolist()
    gini_cols = df.filter(like='gini.').columns.tolist()

    all_cols = cols_to_keep + language_cols + currency_cols + gini_cols

    # Keep only columns that actually exist in the df
    all_cols = [col for col in all_cols if col in df.columns]
    df_clean = df[all_cols].copy()

    # Rename for clarity
    df_clean.rename(columns={
        'name.common': 'country',
        'cca3': 'code',
        'idd.root': 'phone_root',
        'idd.suffixes': 'phone_suffixes'
    }, inplace=True)

    # Drop rows where country name is missing
    df_clean.dropna(subset=['country'], inplace=True)

    # Combine phone root and suffixes into one column
    df_clean['phone_code'] = df_clean['phone_root'] + df_clean['phone_suffixes'].apply(lambda x: x[0] if isinstance(x, list) else '')

    # Drop the separate columns
    df_clean.drop(columns=['phone_root', 'phone_suffixes'], inplace=True)

    # Reset index
    df_clean.reset_index(drop=True, inplace=True)

    print(f"Cleaned DataFrame: {df_clean.shape[0]} countries, {df_clean.shape[1]} columns")
    return df_clean


if __name__ == "__main__":
    countries = fetch_countries_data()
    if countries:
        df = load_to_dataframe(countries)
        df_clean = transform_data(df)



Successfully fetched 59 countries
     tld cca2 ccn3 cca3 cioc  independent               status  unMember  \
0  [.zw]   ZW  716  ZWE  ZIM         True  officially-assigned      True   
1  [.cd]   CD  180  COD  COD         True  officially-assigned      True   
2  [.gh]   GH  288  GHA  GHA         True  officially-assigned      True   
3  [.mu]   MU  480  MUS  MRI         True  officially-assigned      True   
4  [.sc]   SC  690  SYC  SEY         True  officially-assigned      True   

        capital                                       altSpellings  ...  \
0      [Harare]                         [ZW, Republic of Zimbabwe]  ...   
1    [Kinshasa]  [CD, DR Congo, Congo-Kinshasa, Congo, the Demo...  ...   
2       [Accra]                                               [GH]  ...   
3  [Port Louis]  [MU, Republic of Mauritius, République de Maur...  ...   
4    [Victoria]  [SC, Republic of Seychelles, Repiblik Sesel, R...  ...   

  currencies.EGP.symbol currencies.EGP.name  currencies.SS

In [24]:
load_dotenv()

def load_to_mongodb(df_clean):
    client = MongoClient(os.getenv("MONGO_URI"))
    db = client[os.getenv("DB_NAME")]
    collection = db[os.getenv("COLLECTION_NAME")]

    records = df_clean.to_dict('records')
    collection.insert_many(records)

    print(f"Inserted {len(records)} documents into MongoDB")
    client.close()

In [21]:
load_dotenv()

False

In [4]:
for col in df_clean.columns:
    print(col)

country
code
capital
region
subregion
population
area
landlocked
borders
latlng
flag
phone_root
phone_suffixes
languages.bwg
languages.eng
languages.kck
languages.khi
languages.ndc
languages.nde
languages.nya
languages.sna
languages.sot
languages.toi
languages.tsn
languages.tso
languages.ven
languages.xho
languages.zib
languages.fra
languages.kon
languages.lin
languages.lua
languages.swa
languages.mfe
languages.crs
languages.ara
languages.tir
languages.por
languages.spa
languages.zdj
languages.run
languages.amh
languages.kin
languages.afr
languages.nbl
languages.nso
languages.ssw
languages.zul
languages.som
languages.ber
languages.mey
languages.sag
languages.mlg
languages.deu
languages.her
languages.hgm
languages.kwn
languages.loz
languages.ndo
languages.pov
currencies.ZWL.symbol
currencies.ZWL.name
currencies.CDF.symbol
currencies.CDF.name
currencies.GHS.symbol
currencies.GHS.name
currencies.MUR.symbol
currencies.MUR.name
currencies.SCR.symbol
currencies.SCR.name
currencies.ERN.symbol

In [2]:
name = df.filter(like= 'gini.')

print(name)

    gini.2019  gini.2012  gini.2016  gini.2017  gini.2018  gini.2015  \
0        50.3        NaN        NaN        NaN        NaN        NaN   
1         NaN       42.1        NaN        NaN        NaN        NaN   
2         NaN        NaN       43.5        NaN        NaN        NaN   
3         NaN        NaN        NaN       36.8        NaN        NaN   
4         NaN        NaN        NaN        NaN       32.1        NaN   
5         NaN        NaN        NaN        NaN        NaN        NaN   
6         NaN        NaN        NaN        NaN       51.3        NaN   
7         NaN        NaN        NaN        NaN        NaN       41.5   
8         NaN        NaN        NaN        NaN        NaN        NaN   
9         NaN        NaN        NaN       38.0        NaN        NaN   
10        NaN        NaN       42.8        NaN        NaN        NaN   
11        NaN        NaN        NaN        NaN        NaN        NaN   
12        NaN        NaN        NaN        NaN        NaN       

In [5]:
languages = df_clean.filter(like='maps')

print(languages)

Empty DataFrame
Columns: []
Index: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58]


In [16]:
print(df_clean['phone_code'])

0         +263
1         +243
2         +233
3         +230
4         +248
5         +291
6         +244
7         +225
8         +223
9         +241
10        +256
11        +218
12        +227
13        +240
14        +253
15        +246
16        +266
17        +269
18        +222
19        +235
20        +257
21        +251
22        +220
23        +228
24        +232
25        +267
26        +262
27        +221
28        +254
29        +226
30        +250
31        +265
32        +242
33         +27
34        +252
35        +237
36    +2125288
37        +260
38        +249
39        +231
40        +255
41        +213
42        +268
43        +290
44        +236
45        +262
46        +224
47        +239
48        +261
49        +258
50        +212
51        +238
52        +264
53        +234
54         +20
55        +211
56        +245
57        +216
58        +229
Name: phone_code, dtype: object
